# Kiểm tra và trực quan dữ liệu lỗi PCB trên Kaggle

**Bản sửa VOC:** có thể ghép theo tên file XML khi `<filename>` bị cũ, nhưng chỉ khi
ảnh ứng viên duy nhất và kích thước ảnh khớp `<size>` trong XML. Ghi riêng mọi lần
ghép dự phòng; không thay tọa độ hoặc chỉnh sửa XML nguồn.

Notebook khảo sát dữ liệu **trước khi chia tập, tăng cường và huấn luyện**. Hỗ trợ
PASCAL VOC XML, COCO detection JSON và YOLO detection TXT (cần tên lớp đúng thứ tự).
Không tự sửa nhãn, xóa ảnh hoặc chia train/test.

**Cách chạy:** tạo Kaggle Notebook → Import Notebook này → Add Input bộ PCB đã tải →
Settings / Accelerator chọn **GPU T4 ×2** nếu có → chạy lần lượt các cell.
Không cần Internet hoặc cài thêm thư viện trên Kaggle image thông thường.

Nếu chỉ thêm một bộ PCB, giữ `DATA_ROOT=None` để tự dò. Nếu có nhiều input,
đặt `DATA_ROOT` thành thư mục chính xác. ZIP được giải nén vào `/kaggle/working`,
không ghi lên `/kaggle/input`. Dữ liệu dạng RAR/7z cần giải nén trước.

**GPU:** khảo sát ảnh/nhãn dùng CPU và đọc đĩa là chính. Notebook kiểm tra cả hai GPU
và chạy phép tính nhỏ trên từng GPU; không dùng DDP vì chưa huấn luyện mô hình.
Chưa thể khẳng định dữ liệu của bạn là HRIPCB gốc hay bản augmented trước khi khảo sát.


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
import os, json, random, re, hashlib, zipfile, warnings
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
pd.set_option('display.max_colwidth', 100)
print('Thư viện đã sẵn sàng.')


## 1. Cấu hình đường dẫn và định dạng

Chỉ cần sửa `DATA_ROOT` nếu tự dò chọn nhầm thư mục. `ANNOTATION_FORMAT` có thể là
`auto`, `voc`, `coco`, `yolo`. Nếu cùng dữ liệu có nhiều định dạng, chỉ đọc một định dạng
để tránh đếm nhãn hai lần. Khi dùng COCO, file phải chứa bbox detection.

VOC chuẩn dùng tọa độ 1-based inclusive: notebook chuyển sang 0-based half-open.
Nếu nguồn XML của bạn đã dùng 0-based, đặt `VOC_ONE_BASED=False`.
Với YOLO, ưu tiên tên lớp từ YAML; nếu không có YAML, phải điền `CLASS_NAMES` theo
đúng thứ tự ID của nguồn. Không đoán tên lớp từ ID.


In [ ]:
DATA_ROOT = None  # Ví dụ: '/kaggle/input/pcb-defects/PCB_DATASET'
INPUT_BASE = Path('/kaggle/input')
OUT = Path('/kaggle/working/pcb_eda')
OUT.mkdir(parents=True, exist_ok=True)
ANNOTATION_FORMAT = 'auto'
CLASS_NAMES = None  # YOLO: ví dụ ['missing_hole', ...] theo đúng ID nguồn
VOC_ONE_BASED = True
SCAN_ZIP = True
CHECK_HASH = True  # SHA-256: phát hiện file ảnh giống hệt byte; có thể tắt để chạy nhanh
WORKERS = min(8, os.cpu_count() or 2)
NUM_EXAMPLES = 12
MODEL_INPUT = 640
SMALL_EDGE = 32  # Mốc khảo sát: sqrt(diện tích bbox) <32 px SAU letterbox 640
MEDIUM_EDGE = 96


## 2. Kiểm tra T4 ×2 và môi trường


In [ ]:
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA khả dụng:', torch.cuda.is_available())
    gpu_n = torch.cuda.device_count()
    print('Số GPU:', gpu_n)
    for i in range(gpu_n):
        prop = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {prop.name}, VRAM {prop.total_memory / 2**30:.1f} GiB')
        with torch.cuda.device(i), torch.no_grad():
            x = torch.ones((256, 256), device=f'cuda:{i}')
            y = x @ x
            torch.cuda.synchronize(i)
            assert y[0, 0].item() == 256
            del x, y
    if gpu_n != 2:
        print('Chưa thấy đúng 2 GPU. Kiểm tra Accelerator nếu muốn T4 ×2; EDA vẫn chạy được.')
except ImportError:
    print('Không có PyTorch; vẫn có thể chạy toàn bộ phần khảo sát bằng CPU.')


## 3. Dò input và giải nén ZIP nếu cần


In [ ]:
assert INPUT_BASE.exists() or DATA_ROOT, 'Hãy Add Input dataset trên Kaggle trước.'
if DATA_ROOT is None:
    candidates = sorted(p for p in INPUT_BASE.iterdir() if p.is_dir())
    print('Các input tìm thấy:', [str(p) for p in candidates])
    if len(candidates) != 1:
        raise ValueError('Có 0 hoặc nhiều input. Điền DATA_ROOT ở cell cấu hình rồi chạy lại.')
    root = candidates[0]
else:
    root = Path(DATA_ROOT)
assert root.exists(), f'Không tồn tại: {root}'
if root.is_file() and root.suffix.lower() != '.zip':
    raise ValueError('DATA_ROOT phải là thư mục hoặc file ZIP.')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
archives = [root] if root.is_file() else sorted(root.rglob('*.zip'))
has_images = root.is_dir() and any(p.suffix.lower() in IMAGE_EXTS for p in root.rglob('*'))
if SCAN_ZIP and archives and not has_images:
    unpack = OUT / 'extracted'
    unpack.mkdir(exist_ok=True)
    for archive in archives:
        tag = hashlib.sha1(str(archive).encode()).hexdigest()[:10]
        dest = unpack / (archive.stem + '_' + tag)
        marker = dest / '.extraction_complete'
        if not marker.exists():
            dest.mkdir(exist_ok=True)
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    target = (dest / member.filename).resolve()
                    if not target.is_relative_to(dest.resolve()):
                        raise ValueError('ZIP chứa đường dẫn ra ngoài thư mục giải nén.')
                z.extractall(dest)
            marker.touch()
    root = unpack
print('Đang khảo sát:', root)
all_files = sorted(p for p in root.rglob('*') if p.is_file() and '__MACOSX' not in p.parts)
display(pd.DataFrame(Counter(p.suffix.lower() for p in all_files).most_common(),
                     columns=['Phần mở rộng', 'Số file']))
print('Ví dụ đường dẫn:')
for p in all_files[:18]:
    print(' ', p.relative_to(root))


## 4. Đọc ảnh và nhãn về một bảng thống nhất

Ảnh không tìm thấy nhãn được báo là **missing**, không mặc định là ảnh bình thường.
Nhãn rỗng hợp lệ được phân biệt với nhãn lỗi. Các lớp giữ nguyên tên của nguồn,
chỉ bỏ khoảng trắng và thay dấu cách/gạch nối bằng gạch dưới để thống kê.
Các bbox ngoài ảnh hoặc suy biến sẽ được báo, không âm thầm sửa.


In [ ]:
image_paths = [p for p in all_files if p.suffix.lower() in IMAGE_EXTS]
assert image_paths, 'Không tìm thấy ảnh. Kiểm tra DATA_ROOT hoặc giải nén dữ liệu.'
by_name, by_stem = defaultdict(list), defaultdict(list)
for p in image_paths:
    by_name[p.name.casefold()].append(p)
    by_stem[p.stem.casefold()].append(p)

issues, boxes, annotation_counts = [], [], Counter()
labelled, failed_labels = set(), set()
def issue(kind, source, detail):
    issues.append(dict(kind=kind, source=str(source), detail=str(detail)))

def resolve_image(filename, annotation):
    # Ưu tiên đường dẫn chính xác; nếu chỉ có tên, xét độ gần thư mục.
    filename = str(filename).replace('\\', '/')
    p = Path(filename)
    for base in (annotation.parent, annotation.parent.parent, root):
        candidate = base / p
        if candidate in image_paths:
            return candidate
    matches = by_name[p.name.casefold()] or by_stem[p.stem.casefold()]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        def score(candidate):
            return len(Path(os.path.commonpath([str(candidate.parent), str(annotation.parent)])).parts)
        best = max(map(score, matches))
        matches = [m for m in matches if score(m) == best]
        if len(matches) == 1:
            return matches[0]
    raise ValueError(f'Không ghép được ảnh duy nhất: {filename}; số ứng viên={len(matches)}')

def cls_name(x):
    return re.sub(r'[\s-]+', '_', str(x).strip().lower())

def resolve_voc_image(doc, ann):
    declared = doc.findtext('filename')
    try:
        return resolve_image(declared or ann.stem, ann)
    except ValueError as original_error:
        # Chỉ dùng tên XML để khôi phục một liên kết hỏng, không thay thế liên kết đã ghép được.
        img = resolve_image(ann.stem, ann)
        size = doc.find('size')
        if size is None:
            raise ValueError('Ghép dự phòng bị từ chối: XML không có size.') from original_error
        expected = (int(size.findtext('width')), int(size.findtext('height')))
        with Image.open(img) as im:
            actual = im.size
        if actual != expected:
            raise ValueError(f'Ghép dự phòng bị từ chối: XML size={expected}, ảnh={actual}. Không tự scale bbox.') from original_error
        issue('voc_filename_fallback', ann,
              f'filename={declared!r}; dùng ảnh trùng tên XML: {img.name}; size={actual}; không đổi bbox.')
        return img

xmls = [p for p in all_files if p.suffix.lower() == '.xml']
coco_files = []
for p in all_files:
    if p.suffix.lower() == '.json':
        try:
            data = json.loads(p.read_text())
            if isinstance(data, dict) and all(k in data for k in ('images','annotations','categories')):
                coco_files.append(p)
        except (ValueError, OSError):
            pass
fmt = ANNOTATION_FORMAT
if fmt == 'auto':
    fmt = 'voc' if xmls else ('coco' if coco_files else 'yolo')
assert fmt in {'voc','coco','yolo'}
print('Định dạng được chọn:', fmt)

if fmt == 'voc':
    for ann in tqdm(xmls, desc='Đọc VOC XML'):
        img = None
        try:
            doc = ET.parse(ann).getroot()
            img = resolve_voc_image(doc, ann)
            annotation_counts[str(img)] += 1
            labelled.add(str(img))
            for obj in doc.findall('object'):
                b = obj.find('bndbox')
                coords = [float(b.findtext(k)) for k in ('xmin','ymin','xmax','ymax')]
                if VOC_ONE_BASED:
                    coords[0] -= 1
                    coords[1] -= 1
                name = obj.findtext('name')
                if not name or not name.strip():
                    raise ValueError('Tên lớp rỗng')
                boxes.append(dict(image=str(img), annotation=str(ann), class_name=cls_name(name),
                                  x1=coords[0], y1=coords[1], x2=coords[2], y2=coords[3]))
        except Exception as e:
            if img is not None: failed_labels.add(str(img))
            issue('annotation_parse_or_match', ann, e)
elif fmt == 'coco':
    assert coco_files, 'Không tìm thấy COCO detection JSON.'
    for ann in tqdm(coco_files, desc='Đọc COCO JSON'):
        data = json.loads(ann.read_text())
        cats = {c['id']: cls_name(c['name']) for c in data['categories']}
        ids = {}
        for item in data['images']:
            try:
                img = resolve_image(item['file_name'], ann)
                ids[item['id']] = img
                labelled.add(str(img)); annotation_counts[str(img)] += 1
            except Exception as e: issue('annotation_match', ann, e)
        for item in data['annotations']:
            img = ids.get(item.get('image_id'))
            if img is None: continue
            try:
                x, y, w, h = map(float, item['bbox'])
                boxes.append(dict(image=str(img), annotation=str(ann), class_name=cats[item['category_id']],
                                  x1=x, y1=y, x2=x+w, y2=y+h))
                if item.get('iscrowd', 0): issue('coco_crowd', ann, 'Có iscrowd=1; cần quy tắc riêng khi train.')
            except Exception as e:
                failed_labels.add(str(img)); issue('annotation_parse', ann, e)
else:
    # Đọc class names từ YAML khi chỉ có duy nhất một ánh xạ.
    mappings = []
    try:
        import yaml
        for p in all_files:
            if p.suffix.lower() in {'.yaml', '.yml'}:
                d = yaml.safe_load(p.read_text())
                if isinstance(d, dict) and 'names' in d:
                    n = d['names']
                    if isinstance(n, dict):
                        n = {int(k): v for k,v in n.items()}
                        n = [n[i] for i in range(len(n))]
                    if isinstance(n, list) and n not in mappings: mappings.append(n)
    except ImportError:
        pass
    if CLASS_NAMES is None and len(mappings) == 1: CLASS_NAMES = mappings[0]
    assert CLASS_NAMES, 'YOLO: điền CLASS_NAMES đúng thứ tự ID nguồn, hoặc cung cấp data.yaml.'
    txts = [p for p in all_files if p.suffix.lower() == '.txt' and
            ('labels' in [s.lower() for s in p.parts] or p.stem.casefold() in by_stem)]
    for ann in tqdm(txts, desc='Đọc YOLO TXT'):
        img = None
        try:
            parts = list(ann.parts)
            idx = next((i for i,s in enumerate(parts) if s.lower() == 'labels'), None)
            if idx is not None:
                parts[idx] = 'images'
                counterpart = Path(*parts)
                matches = [p for p in by_stem[ann.stem.casefold()] if p.parent == counterpart.parent]
                img = matches[0] if len(matches) == 1 else resolve_image(ann.stem, ann)
            else: img = resolve_image(ann.stem, ann)
            labelled.add(str(img)); annotation_counts[str(img)] += 1
            with Image.open(img) as im: W,H = im.size
            for line in ann.read_text().splitlines():
                if not line.strip(): continue
                fields = line.split()
                if len(fields) != 5: raise ValueError('Chỉ hỗ trợ YOLO detection: class cx cy w h.')
                cid, cx, cy, w, h = map(float, fields)
                if not cid.is_integer() or not 0 <= cid < len(CLASS_NAMES): raise ValueError('Class ID không hợp lệ')
                if not all(np.isfinite([cx,cy,w,h])) or not all(0<=v<=1 for v in (cx,cy,w,h)):
                    raise ValueError('Tọa độ YOLO phải hữu hạn và chuẩn hóa [0,1]')
                boxes.append(dict(image=str(img), annotation=str(ann), class_name=cls_name(CLASS_NAMES[int(cid)]),
                                  x1=(cx-w/2)*W,y1=(cy-h/2)*H,x2=(cx+w/2)*W,y2=(cy+h/2)*H))
        except Exception as e:
            if img is not None: failed_labels.add(str(img))
            issue('annotation_parse_or_match', ann, e)
for img, n in annotation_counts.items():
    if n > 1:
        failed_labels.add(img)
        issue('multiple_annotation_files', img, f'{n} nguồn nhãn cho cùng ảnh; không gộp âm thầm.')
box_df = pd.DataFrame(boxes, columns=['image','annotation','class_name','x1','y1','x2','y2'])
print('Số bbox đọc được:', len(box_df))


## 5. Kiểm tra toàn bộ ảnh, tọa độ và nhãn trùng


In [ ]:
def inspect_image(p):
    row = dict(image=str(p), filename=p.name, width=np.nan, height=np.nan,
               mode='', readable=False, sha256='', error='')
    try:
        with Image.open(p) as im:
            im.load()  # Giải mã toàn ảnh để phát hiện file hỏng
            row.update(width=im.width,height=im.height,mode=im.mode,readable=True)
        if CHECK_HASH:
            h = hashlib.sha256()
            with p.open('rb') as f:
                for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
            row['sha256'] = h.hexdigest()
    except Exception as e: row['error'] = str(e)
    return row

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    images_df = pd.DataFrame(tqdm(pool.map(inspect_image,image_paths),total=len(image_paths),desc='Kiểm tra ảnh'))
images_df['label_status'] = images_df.image.map(lambda p: 'error' if p in failed_labels else ('present' if p in labelled else 'missing'))
images_df['split_hint'] = images_df.image.map(lambda p: next((s.lower() for s in Path(p).relative_to(root).parts[:-1]
                    if s.lower() in {'train','test','val','valid','validation','train2017','val2017'}), 'unknown'))
for r in images_df.loc[~images_df.readable].itertuples(): issue('unreadable_image',r.image,r.error)
for r in images_df.loc[images_df.label_status == 'missing'].itertuples(): issue('missing_label',r.image,'Chưa thấy nhãn; không mặc định là ảnh âm tính.')

box_df = box_df.merge(images_df[['image','width','height','readable','label_status']],on='image',how='left')
coords = box_df[['x1','y1','x2','y2']].to_numpy(dtype=float)
box_df['valid'] = (np.isfinite(coords).all(axis=1) & box_df.readable.fillna(False)
    & (box_df.label_status == 'present') & (box_df.x2 > box_df.x1) & (box_df.y2 > box_df.y1)
    & (box_df.x1 >= 0) & (box_df.y1 >= 0) & (box_df.x2 <= box_df.width) & (box_df.y2 <= box_df.height))
for r in box_df.loc[~box_df.valid].itertuples():
    issue('invalid_bbox_or_label',r.annotation,f'{Path(r.image).name}: {r.x1,r.y1,r.x2,r.y2}, size={r.width,r.height}')
key = ['image','class_name','x1','y1','x2','y2']
box_df['duplicate_bbox'] = box_df.duplicated(key,keep='first')
for r in box_df.loc[box_df.duplicate_bbox].itertuples(): issue('duplicate_bbox',r.annotation,r.image)
valid = box_df.loc[box_df.valid & ~box_df.duplicate_bbox].copy()
counts = valid.groupby('image').size()
images_df['valid_boxes'] = images_df.image.map(counts).fillna(0).astype(int)
images_df['confirmed_empty'] = ((images_df.label_status=='present') & images_df.readable &
    ~images_df.image.isin(box_df.image))
duplicates = images_df.loc[images_df.sha256.ne('') & images_df.sha256.duplicated(keep=False)].copy()
display(images_df.head())
print('Nhãn present nhưng mọi bbox đều lỗi cũng KHÔNG được coi là ảnh bình thường.')


## 6. Tổng quan và nhận diện đặc điểm bản dữ liệu


In [ ]:
summary = {
    'root': str(root), 'format':fmt, 'images_total':len(images_df),
    'images_readable':int(images_df.readable.sum()), 'images_label_missing':int((images_df.label_status=='missing').sum()),
    'images_label_error':int((images_df.label_status=='error').sum()),
    'confirmed_empty_images':int(images_df.confirmed_empty.sum()),
    'raw_boxes':len(box_df), 'usable_unique_boxes':len(valid),
    'classes':sorted(valid.class_name.unique().tolist()),
    'duplicate_image_files':len(duplicates), 'hash_checked':CHECK_HASH,
    'voc_filename_fallbacks':sum(i['kind']=='voc_filename_fallback' for i in issues),
}
display(pd.DataFrame([(k,v) for k,v in summary.items() if k!='classes'],columns=['Mục','Giá trị']))
print('Tên lớp:', summary['classes'])
print('Phân bố split theo thư mục (chỉ là gợi ý):')
display(images_df.groupby(['split_hint','label_status']).size().unstack(fill_value=0))
sizes = images_df.loc[images_df.readable].groupby(['width','height']).size().sort_values(ascending=False)
display(sizes.head(15).rename('Số ảnh').to_frame())
if len(sizes) and sizes.iloc[0]/max(1,images_df.readable.sum()) > 0.8 and sizes.index[0] in [(600,600),(640,640)]:
    print('Đa số ảnh là 600/640 vuông: có thể là bản crop/resize. Không đủ bằng chứng kết luận nguồn gốc.')
print('Không dùng riêng con số 1.386 hay 10.668 để kết luận đây là bản gốc/augmented.')
assert len(valid), 'Không có bbox hợp lệ. Xem issues ở cell 11 và cấu hình nhãn trước khi trực quan.'


## 7. Phân bố lớp và số lỗi trên mỗi ảnh


In [ ]:
classes = sorted(valid.class_name.unique())
palette = {c:plt.get_cmap('tab20')(i % 20) for i,c in enumerate(classes)}
class_stats = valid.groupby('class_name').agg(boxes=('image','size'),images=('image','nunique')).sort_values('boxes',ascending=False)
class_stats['percent_boxes'] = 100*class_stats.boxes/class_stats.boxes.sum()
display(class_stats.round(2))
fig, axs = plt.subplots(1,2,figsize=(14,5))
bars = axs[0].barh(class_stats.index[::-1],class_stats.boxes.iloc[::-1],color=[palette[c] for c in class_stats.index[::-1]])
axs[0].bar_label(bars,padding=3); axs[0].set_title('Số vùng lỗi theo lớp'); axs[0].set_xlabel('Bounding boxes')
eligible = images_df.loc[images_df.readable & images_df.label_status.eq('present')]
axs[1].hist(eligible.valid_boxes,bins=min(30,max(1,int(eligible.valid_boxes.max())+1)),color='#377eb8',edgecolor='white')
axs[1].set(title='Số bbox hợp lệ trên mỗi ảnh có nhãn',xlabel='Số bbox',ylabel='Số ảnh')
fig.tight_layout(); fig.savefig(OUT/'01_class_distribution.png',bbox_inches='tight'); plt.show()
ratio = class_stats.boxes.max()/class_stats.boxes.min()
print(f'Tỷ lệ lớp nhiều/ít bbox nhất: {ratio:.2f}. Đây chưa phải thước đo độ khó của lớp.')


## 8. Kích thước ảnh, bbox và lỗi nhỏ sau khi thu ảnh

Giả lập **letterbox giữ tỷ lệ, không scale-up**, cạnh đầu vào `MODEL_INPUT`.
Đây là quy ước khảo sát; hãy đặt lại theo pipeline huấn luyện thực tế.
Nhóm small/medium/large dựa trên diện tích bbox sau phép đổi tỷ lệ này,
**không phải kết quả AP-small của COCO evaluator**.


In [ ]:
valid['bbox_w'] = valid.x2-valid.x1
valid['bbox_h'] = valid.y2-valid.y1
valid['relative_area'] = valid.bbox_w*valid.bbox_h/(valid.width*valid.height)
valid['aspect_ratio'] = valid.bbox_w/valid.bbox_h
valid['resize_scale'] = np.minimum(1.0, MODEL_INPUT/valid[['width','height']].max(axis=1))
valid['bbox_w_input'] = valid.bbox_w*valid.resize_scale
valid['bbox_h_input'] = valid.bbox_h*valid.resize_scale
valid['edge_input'] = np.sqrt(valid.bbox_w_input*valid.bbox_h_input)
valid['size_group'] = pd.cut(valid.edge_input, bins=[0,SMALL_EDGE,MEDIUM_EDGE,np.inf],
                            labels=['small','medium','large'],right=False)
size_by_class = pd.crosstab(valid.class_name,valid.size_group).reindex(columns=['small','medium','large'],fill_value=0)
size_by_class['small_percent'] = 100*size_by_class['small']/size_by_class[['small','medium','large']].sum(axis=1)
display(size_by_class.round(2))
fig, axs = plt.subplots(2,2,figsize=(14,10))
r = images_df[images_df.readable]
axs[0,0].scatter(r.width,r.height,alpha=.3,s=15)
axs[0,0].set(title='Kích thước ảnh',xlabel='Width (px)',ylabel='Height (px)')
axs[0,1].scatter(valid.bbox_w_input,valid.bbox_h_input,alpha=.2,s=10)
axs[0,1].set(title=f'BBox sau letterbox {MODEL_INPUT}',xlabel='BBox width (px)',ylabel='BBox height (px)')
axs[1,0].hist(valid.relative_area*100,bins=50,color='#4daf4a')
axs[1,0].set(title='Diện tích bbox / diện tích ảnh',xlabel='Tỷ lệ diện tích (%)',ylabel='Số bbox')
size_by_class[['small','medium','large']].plot.barh(stacked=True,ax=axs[1,1],color=['#e41a1c','#ffb74d','#377eb8'])
axs[1,1].set(title='Nhóm kích thước theo lớp',xlabel='Số bbox',ylabel='')
fig.tight_layout();fig.savefig(OUT/'02_sizes.png',bbox_inches='tight');plt.show()
tiny = valid[(valid.bbox_w_input<4)|(valid.bbox_h_input<4)]
print(f'{len(tiny)}/{len(valid)} bbox có ít nhất một cạnh dưới 4 px sau resize giả lập.')
print('Đây là nhóm cần xem kỹ khi thiết kế crop; không khẳng định mô hình chắc chắn bỏ sót.')


## 9. Trực quan ảnh ngẫu nhiên và ảnh theo từng lớp


In [ ]:
def show_images(paths, cols=3, title='', filename=None):
    paths = list(paths)
    if not paths:
        print('Không có ảnh phù hợp.'); return
    rows = (len(paths)+cols-1)//cols
    fig, axes = plt.subplots(rows,cols,figsize=(cols*5,rows*4.4),squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax,p in zip(axes.flat,paths):
        with Image.open(p) as im: ax.imshow(im.convert('RGB'))
        subset = valid[valid.image.eq(str(p))]
        for b in subset.itertuples():
            color=palette[b.class_name]
            ax.add_patch(Rectangle((b.x1,b.y1),b.x2-b.x1,b.y2-b.y1,fill=False,edgecolor=color,linewidth=1.5))
            ax.text(b.x1,b.y1,b.class_name,color='black',fontsize=7,va='bottom',
                    bbox=dict(facecolor=color,alpha=.85,pad=1,edgecolor='none'))
        ax.set_title(f'{Path(p).name} | {len(subset)} bbox',fontsize=9)
    fig.suptitle(title,fontsize=14); fig.tight_layout(rect=[0,0,1,.96])
    if filename: fig.savefig(OUT/filename,bbox_inches='tight')
    plt.show()

pool = sorted(valid.image.unique())
chosen = random.Random(SEED).sample(pool,min(NUM_EXAMPLES,len(pool)))
show_images(chosen,title='Ảnh ngẫu nhiên có bounding box',filename='03_random_examples.png')


In [ ]:
# Mỗi lớp chọn một ảnh; ảnh có thể lặp nếu chứa nhiều lớp.
per_class = [random.Random(SEED+i).choice(sorted(valid.loc[valid.class_name.eq(c),'image'].unique()))
             for i,c in enumerate(classes)]
show_images(per_class,title='Ví dụ phủ các lớp lỗi',filename='04_class_examples.png')


## 10. Xem cận cảnh lỗi nhỏ và heatmap vị trí

Các crop dưới đây **chỉ để quan sát**, chưa tạo dữ liệu huấn luyện.
Ảnh hiển thị có thể được phóng to trên màn hình; không có thêm thông tin ảnh thật.


In [ ]:
smallest = valid.nsmallest(12,'edge_input')
fig, axes = plt.subplots(3,4,figsize=(16,11),squeeze=False)
for ax in axes.flat: ax.axis('off')
for ax, b in zip(axes.flat,smallest.itertuples()):
    with Image.open(b.image) as im:
        margin = max(24, max(b.bbox_w,b.bbox_h)*1.5)
        left=max(0,int(np.floor(b.x1-margin))); top=max(0,int(np.floor(b.y1-margin)))
        right=min(im.width,int(np.ceil(b.x2+margin))); bottom=min(im.height,int(np.ceil(b.y2+margin)))
        ax.imshow(im.crop((left,top,right,bottom)).convert('RGB'))
    ax.add_patch(Rectangle((b.x1-left,b.y1-top),b.bbox_w,b.bbox_h,
                          fill=False,edgecolor=palette[b.class_name],linewidth=2))
    ax.set_title(f'{b.class_name}\nGốc {b.bbox_w:.0f}×{b.bbox_h:.0f} → {b.bbox_w_input:.1f}×{b.bbox_h_input:.1f}px',fontsize=9)
fig.tight_layout();fig.savefig(OUT/'05_small_defect_crops.png',bbox_inches='tight');plt.show()

cx=(valid.x1+valid.x2)/(2*valid.width)
cy=(valid.y1+valid.y2)/(2*valid.height)
fig,ax=plt.subplots(figsize=(7,6))
h=ax.hist2d(cx,cy,bins=32,range=[[0,1],[0,1]],cmap='magma')
ax.invert_yaxis();ax.set(title='Phân bố tâm bbox trên ảnh',xlabel='x / width',ylabel='y / height')
fig.colorbar(h[3],ax=ax,label='Số bbox');fig.tight_layout()
fig.savefig(OUT/'06_bbox_heatmap.png',bbox_inches='tight');plt.show()


## 11. Báo cáo lỗi và ảnh trùng

SHA-256 chỉ phát hiện ảnh **giống hệt byte**. Nó không chứng minh không có rò rỉ giữa
ảnh xoay, crop, nén khác nhau, hoặc ảnh chung bảng mạch/template. Trước khi chia tập,
cần xác nhận ID ảnh nguồn/board và nhóm các biến thể vào cùng một tập.

`voc_filename_fallback` là nhật ký khôi phục cách ghép tên, không phải lỗi thiếu nhãn.
Các hàng này vẫn cần kiểm tra trực quan mẫu. Nhãn với tọa độ sai vẫn bị báo như trước.


In [ ]:
issues_df = pd.DataFrame(issues,columns=['kind','source','detail'])
fallback_df = issues_df.loc[issues_df.kind.eq('voc_filename_fallback')].copy()
if len(fallback_df):
    print(f'Đã khôi phục {len(fallback_df)} liên kết bằng tên XML + kiểm tra size. Xem mẫu bên dưới:')
    recovered_annotations = set(fallback_df.source)
    recovered_paths = sorted(valid.loc[valid.annotation.isin(recovered_annotations),'image'].unique())
    show_images(recovered_paths[:6],title='Mẫu khôi phục liên kết ảnh–nhãn',filename='07_recovered_label_examples.png')
if len(issues_df):
    display(issues_df.groupby('kind').size().rename('Số vấn đề').to_frame())
    display(issues_df.head(30))
else: print('Không phát hiện vấn đề theo các kiểm tra hiện có.')
if len(duplicates):
    print('Ảnh trùng SHA-256 (chưa tự xóa):')
    display(duplicates[['image','sha256','split_hint']].head(30))
else: print('Không thấy file ảnh trùng byte.' if CHECK_HASH else 'Chưa kiểm tra hash.')
print('Tên file mẫu để nhận diện quy tắc board/source:')
display(images_df[['filename','split_hint','label_status']].head(25))


## 12. Xuất kết quả khảo sát và gợi ý bước tiếp theo


In [ ]:
images_df.to_csv(OUT/'images_inventory.csv',index=False)
box_df.to_csv(OUT/'all_boxes_audit.csv',index=False)
valid.to_csv(OUT/'valid_boxes_features.csv',index=False)
issues_df.to_csv(OUT/'issues.csv',index=False)
fallback_df.to_csv(OUT/'voc_filename_fallbacks.csv',index=False)
duplicates.to_csv(OUT/'duplicate_images.csv',index=False)
class_stats.to_csv(OUT/'class_statistics.csv')
size_by_class.to_csv(OUT/'size_by_class.csv')
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Kết quả tại:', OUT)
for p in sorted(OUT.iterdir()):
    if p.is_file(): print(f'{p.name:32s} {p.stat().st_size/1024:.1f} KB')
print('\nLớp có tỷ lệ bbox nhỏ cao nhất (chưa phải lớp khó nhất):')
display(size_by_class.sort_values('small_percent',ascending=False).head(6))
print('Bước tiếp: xử lý lỗi nhãn → xác minh nhóm ảnh nguồn → chia tập cố định → baseline → crop chỉ trên train.')
print('Độ khó theo mô hình chỉ xác định sau baseline, qua AP/Recall từng lớp trên validation.')


**Sau khi chạy:** gửi bảng tổng quan, bảng lớp và bảng kích thước để chốt cấu hình
chia tập/crop. Nếu còn cảnh báo ghép ảnh hoặc thiếu nhãn, kiểm tra `issues.csv` trước.
`valid_boxes_features.csv` là bảng khảo sát, chưa phải bộ dữ liệu YOLO đã làm sạch.

Phần EDA chưa sửa dữ liệu. Tiếp tục cell 13–18 bên dưới để đối chiếu trùng và chia tập.
Khi xuất notebook Kaggle, chọn Save Version để giữ output khảo sát.


## 13. Đối chiếu ảnh trùng và chia tập cố định

Chạy sau phần khảo sát phía trên, trong cùng kernel. Phần này dùng `images_df`,
`box_df`, `valid` sau khi đã ghép đủ nhãn. Không xóa hay sửa file nguồn.

Quy trình: so sánh nhãn của ảnh cùng SHA-256 → chỉ giữ một bản nếu nhãn thống nhất →
gộp nhóm theo bảng mạch và quan hệ trùng byte → chia khoảng 70/15/15 theo nhóm →
kiểm tra ảnh, nhóm, hash và phân bố lớp. Xuất manifest để dùng chung cho mọi mô hình.

Nếu một nhóm ảnh trùng có nhãn khác nhau, mặc định **cách ly toàn bộ nhóm đó khỏi
manifest**, không chọn đại một nhãn. Các file gốc vẫn nguyên vẹn để kiểm tra sau.
Đặt `CONFLICT_POLICY='stop'` nếu muốn bắt buộc xử lý thủ công trước khi chia.

**Cách nhóm mặc định:** `light_01_...`, `l_light_01_...`, `rotation_90_light_01_...`
đều thuộc board `01`, kể cả khi khác loại lỗi hoặc khác crop. Đây là quy tắc suy từ
tên file, cần xác nhận ID thực sự đại diện cùng bảng mạch. Bảo thủ hơn chỉ bỏ tiền tố
xoay rồi chia theo crop, vì nhiều crop có thể lấy từ cùng bảng mạch.

Tỷ lệ số ảnh chỉ gần mục tiêu do không tách nhóm. Validation/test giữ nguyên các mẫu
đã có trong bộ augmented; không tạo thêm augmentation ở bước này.


In [ ]:
from itertools import product
from collections import defaultdict
import hashlib, json, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPLIT_SEED = 42
TARGET_RATIOS = np.array([0.70,0.15,0.15],dtype=float)
SPLIT_NAMES = ['train','val','test']
LABEL_ATOL = 1e-6  # Pixel; khác 1 pixel vẫn báo xung đột, không tự hợp nhất.
CONFLICT_POLICY = 'quarantine'  # 'quarantine' hoặc 'stop'
BOARD_REGEX = r'(?:^|_)light_(\d+)_'  # Kiểm tra bảng ánh xạ ở cell 15.
SPLIT_OUT = Path('/kaggle/working/pcb_prepare')
SPLIT_OUT.mkdir(parents=True,exist_ok=True)
assert np.all(TARGET_RATIOS>0) and np.isclose(TARGET_RATIOS.sum(),1)
assert CONFLICT_POLICY in {'quarantine','stop'}
assert 'images_df' in globals() and 'valid' in globals(), 'Chạy các cell EDA trước.'
assert images_df.readable.all(), 'Còn ảnh không đọc được.'
assert images_df.label_status.eq('present').all(), 'Chạy bản EDA đã sửa để ghép đủ nhãn.'
assert box_df.valid.all(), 'Còn bbox hoặc nhãn lỗi; kiểm tra all_boxes_audit.csv trước.'
assert images_df.image.is_unique, 'Inventory có đường dẫn ảnh lặp.'
assert images_df.sha256.notna().all() and images_df.sha256.ne('').all(), 'Bật CHECK_HASH=True và chạy lại EDA.'
print('Đủ điều kiện đối chiếu và chuẩn bị chia tập.')


## 14. Đối chiếu nhãn trong từng nhóm ảnh trùng

Chỉ so sánh ảnh giống hệt byte, không suy biến đổi bbox từ ảnh xoay. Hai nhãn được
coi là thống nhất khi có cùng lớp, số bbox và tọa độ (sai số cho phép rất nhỏ).
Ảnh giống nhau nhưng nhãn khác nhau sẽ xuất vào `duplicate_label_conflicts.csv`.
Mặc định cách ly cả nhóm; nếu chọn chính sách `stop` thì dừng. Không lấy hợp tất cả
bbox vì có thể tạo nhãn sai. Sai khác tọa độ nhỏ cũng được báo để tránh đoán nhãn.


In [ ]:
def same_annotations(a, b, atol=LABEL_ATOL):
    # Ghép từng bbox trong cùng lớp, không phụ thuộc thứ tự object trong XML.
    if len(a)!=len(b): return False
    if sorted(a.class_name.tolist())!=sorted(b.class_name.tolist()): return False
    cols=['x1','y1','x2','y2']
    for cls in sorted(set(a.class_name)):
        left=a.loc[a.class_name.eq(cls),cols].to_numpy(float)
        right=b.loc[b.class_name.eq(cls),cols].to_numpy(float)
        if len(left)!=len(right): return False
        # Ghép hai phía; tránh phụ thuộc thứ tự khi có bbox rất gần nhau.
        matches=np.all(np.abs(left[:,None,:]-right[None,:,:])<=atol,axis=2)
        assigned={}
        def augment(i,seen):
            for j in np.flatnonzero(matches[i]):
                j=int(j)
                if j in seen: continue
                seen.add(j)
                if j not in assigned or augment(assigned[j],seen):
                    assigned[j]=i
                    return True
            return False
        if not all(augment(i,set()) for i in range(len(left))): return False
    return True

def preferred_image(path):
    name=Path(path).name
    return (0 if name.startswith('light_') else 1,name,str(path))

labels_by_image={p:g.copy() for p,g in valid.groupby('image',sort=False)}
empty_boxes=valid.iloc[:0].copy()
decisions=[]; conflicts=[]
for sha, group in images_df.groupby('sha256',sort=True):
    paths=sorted(group.image.tolist(),key=preferred_image)
    representative=paths[0]
    ref=labels_by_image.get(representative,empty_boxes)
    for path in paths:
        agrees=same_annotations(ref,labels_by_image.get(path,empty_boxes))
        decisions.append(dict(image=path,sha256=sha,representative=representative,
                              keep=(path==representative),label_agrees=agrees,group_size=len(paths)))
        if not agrees:
            conflicts.append(dict(sha256=sha,representative=representative,image=path,
                                  representative_boxes=len(ref),image_boxes=len(labels_by_image.get(path,empty_boxes))))
dedup_df=pd.DataFrame(decisions)
conflicts_df=pd.DataFrame(conflicts,columns=['sha256','representative','image','representative_boxes','image_boxes'])
conflict_hashes=set(conflicts_df.sha256)
dedup_df['quarantined']=dedup_df.sha256.isin(conflict_hashes)
dedup_df.loc[dedup_df.quarantined,'keep']=False
dedup_df.to_csv(SPLIT_OUT/'dedup_decisions.csv',index=False)
conflicts_df.to_csv(SPLIT_OUT/'duplicate_label_conflicts.csv',index=False)
quarantine_df=dedup_df.loc[dedup_df.quarantined].copy()
quarantine_df.to_csv(SPLIT_OUT/'quarantined_images.csv',index=False)
valid.loc[valid.image.isin(quarantine_df.image)].to_csv(SPLIT_OUT/'quarantined_boxes.csv',index=False)
display(conflicts_df.head(20))
print('Ảnh thuộc nhóm trùng:',int(dedup_df.group_size.gt(1).sum()))
print('Số bản dư nhãn thống nhất:',int((~dedup_df.keep & ~dedup_df.quarantined).sum()))
print('Số ảnh cách ly do xung đột nhãn:',len(quarantine_df))
assert conflicts_df.empty or CONFLICT_POLICY=='quarantine', ('Ảnh trùng có nhãn khác nhau. DỪNG: gửi duplicate_label_conflicts.csv '
                           'để đối chiếu, không tự chọn hoặc gộp nhãn.')
clean_images=images_df.merge(dedup_df[['image','representative','keep']],on='image',validate='one_to_one')
clean_images=clean_images.loc[clean_images.keep].copy()
clean_boxes=valid.loc[valid.image.isin(clean_images.image)].copy()
assert clean_images.sha256.is_unique
print('Giữ lại',len(clean_images),'ảnh và',len(clean_boxes),'bbox; chưa thay đổi file nguồn.')


## 15. Nhóm ảnh theo bảng mạch và các mối liên hệ trùng

Code không tự đoán ID nếu tên không theo mẫu: sẽ dừng và in tên cần kiểm tra.
Các board có ảnh trùng byte được nối thành cùng một nhóm, kể cả khi bản dư đã bị
loại khỏi manifest. Việc này tránh mất dấu liên hệ giữa hai board sau dedup.


In [ ]:
def extract_board(filename):
    m=re.search(BOARD_REGEX,filename)
    return 'board_'+m.group(1) if m else None

mapping=images_df[['image','filename','sha256']].copy()
mapping['board_id']=mapping.filename.map(extract_board)
unknown=mapping[mapping.board_id.isna()]
if len(unknown): display(unknown.head(20))
assert unknown.empty, 'Tên file chưa khớp BOARD_REGEX; xác nhận quy tắc ID trước khi chia.'

class UnionFind:
    def __init__(self,keys): self.parent={k:k for k in keys}
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]
            x=self.parent[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a!=b: self.parent[max(a,b)]=min(a,b)

uf=UnionFind(sorted(mapping.board_id.unique()))
for _,group in mapping.groupby('sha256',sort=True):
    ids=sorted(group.board_id.unique())
    for board in ids[1:]: uf.union(ids[0],board)
mapping['group_id']=mapping.board_id.map(uf.find)
mapping.to_csv(SPLIT_OUT/'all_image_groups.csv',index=False)
clean_images=clean_images.merge(mapping[['image','board_id','group_id']],on='image',validate='one_to_one')
group_summary=clean_images.groupby('group_id').agg(images=('image','size'),boards=('board_id',lambda s:','.join(sorted(s.unique()))))
display(group_summary)
print('Số board:',mapping.board_id.nunique(),'| Số nhóm độc lập:',len(group_summary))
assert len(group_summary)>=3, 'Không đủ 3 nhóm độc lập để chia train/val/test.'
print('Cần xác nhận board ID từ tên file đúng nguồn ảnh. Hash không phát hiện mọi ảnh gần trùng.')


## 16. Chọn cách chia theo nhóm với seed cố định

Tìm cách chia gần tỷ lệ mục tiêu, đồng thời cân đối phân bố bbox mỗi lớp và giữ đủ
lớp trong cả ba tập. Không dùng kết quả mô hình để chọn test. Với ≤10 nhóm, duyệt hết
các cách gán; nhiều hơn thì xét 12.000 phương án ngẫu nhiên theo seed cố định.
Nếu không tìm được cách phủ đủ lớp, dừng thay vì tách nhóm để ép tỷ lệ.


In [ ]:
def choose_group_split(image_counts, class_counts, ratios, seed=42, trials=12000):
    n=len(image_counts)
    assert class_counts.shape[0]==n
    totals=class_counts.sum(axis=0)
    assert (totals>0).all()
    candidates=(product(range(3),repeat=n) if n<=10 else
                (np.random.default_rng(seed+i).choice(3,size=n,p=ratios) for i in range(trials)))
    best=None; best_score=np.inf
    for candidate in candidates:
        a=np.asarray(candidate,dtype=int)
        if len(set(a))<3: continue
        counts=np.array([class_counts[a==k].sum(axis=0) for k in range(3)])
        if not (counts>0).all(): continue
        proportions=np.array([image_counts[a==k].sum() for k in range(3)])/image_counts.sum()
        class_proportions=counts/totals[None,:]
        score=float(np.abs(proportions-ratios).mean()+
                    np.abs(class_proportions-ratios[:,None]).mean())
        if score<best_score-1e-12:
            best_score=score;best=a.copy()
    if best is None:
        raise ValueError('Không tìm được cách chia nhóm với đủ lớp ở 3 tập. Xem class_group_counts.csv; không chia ngẫu nhiên từng ảnh.')
    return best,best_score

classes_split=sorted(clean_boxes.class_name.unique())
group_ids=sorted(clean_images.group_id.unique())
# Seed quyết định thứ tự xử lý nhóm để chọn cố định giữa các phương án đồng điểm.
group_ids=list(np.random.default_rng(SPLIT_SEED).permutation(group_ids))
box_groups=clean_boxes.merge(clean_images[['image','group_id']],on='image',validate='many_to_one')
class_group=pd.crosstab(box_groups.group_id,box_groups.class_name).reindex(index=group_ids,columns=classes_split,fill_value=0)
class_group.to_csv(SPLIT_OUT/'class_group_counts.csv')
image_counts=clean_images.groupby('group_id').size().reindex(group_ids).to_numpy()
assignment,split_score=choose_group_split(image_counts,class_group.to_numpy(),TARGET_RATIOS,SPLIT_SEED)
split_map={g:SPLIT_NAMES[int(k)] for g,k in zip(group_ids,assignment)}
manifest=clean_images.copy()
manifest['split']=manifest.group_id.map(split_map)
manifest=manifest.sort_values(['split','group_id','filename']).reset_index(drop=True)
manifest_boxes=clean_boxes.merge(manifest[['image','split','group_id','board_id']],on='image',validate='many_to_one')
print('Điểm lệch tỷ lệ (nhỏ hơn tốt hơn):',round(split_score,5))
print('Phương án được chọn:')
display(manifest.groupby(['split','group_id']).size().rename('images').to_frame())


## 17. Kiểm tra tách biệt và trực quan phân bố sau chia


In [ ]:
assert manifest.image.is_unique and len(manifest)==len(clean_images)
assert manifest['split'].notna().all()
for column in ['image','sha256','group_id','board_id']:
    assert manifest.groupby(column)['split'].nunique().max()==1, f'Rò rỉ theo {column}'
# Cả ảnh bị loại cũng phải thuộc cùng split với đại diện và board của nó.
full_mapping=mapping.merge(dedup_df[['image','representative','keep']],on='image',validate='one_to_one')
full_mapping['split']=full_mapping.group_id.map(split_map)
full_mapping['excluded']=~full_mapping.keep
full_mapping['quarantined']=full_mapping.sha256.isin(conflict_hashes)
full_mapping['group_assigned_split']=full_mapping['split']
full_mapping.loc[full_mapping.quarantined,'split']='quarantine'
assert full_mapping.groupby('sha256')['split'].nunique().max()==1
assert len(manifest_boxes)==len(clean_boxes)

split_summary=manifest.groupby('split').agg(images=('image','size'),groups=('group_id','nunique'),boards=('board_id','nunique')).reindex(SPLIT_NAMES)
split_summary['boxes']=manifest_boxes.groupby('split').size().reindex(SPLIT_NAMES)
split_summary['actual_percent']=100*split_summary.images/len(manifest)
split_summary['target_percent']=100*TARGET_RATIOS
class_split=pd.crosstab(manifest_boxes.class_name,manifest_boxes['split']).reindex(index=classes_split,columns=SPLIT_NAMES,fill_value=0)
assert (class_split>0).all().all(), 'Có lớp thiếu trong một tập.'
display(split_summary.round(2));display(class_split)
fig,axs=plt.subplots(1,2,figsize=(15,5))
bars=axs[0].bar(SPLIT_NAMES,split_summary.images,color=['#377eb8','#ff9f40','#4daf4a'])
axs[0].bar_label(bars,padding=3);axs[0].set(title='Số ảnh sau dedup và chia theo nhóm',ylabel='Số ảnh')
class_split.plot.barh(ax=axs[1]);axs[1].set(title='Số bbox mỗi lớp trong từng tập',xlabel='Số bbox',ylabel='')
fig.tight_layout();fig.savefig(SPLIT_OUT/'split_distribution.png',bbox_inches='tight');plt.show()
print('Đạt kiểm tra: không chung đường dẫn, hash, group hoặc board giữa các tập theo quy tắc đã dùng.')
print('Đây không phải bằng chứng loại trừ mọi ảnh gần trùng nếu ID nguồn bị đặt sai.')


## 18. Lưu manifest cố định và tải báo cáo

`split_manifest.csv` là danh sách ảnh được giữ. `split_boxes.csv` chứa bbox của các
ảnh đó, vẫn ở tọa độ pixel 0-based half-open như EDA. Chưa xuất nhãn YOLO hay train.
Các file danh sách train/val/test dùng đường dẫn input của phiên Kaggle; cần remap
nếu đổi nơi mount dataset. Không tạo crop/augmentation trước khi chốt manifest.


In [ ]:
manifest.to_csv(SPLIT_OUT/'split_manifest.csv',index=False)
manifest_boxes.to_csv(SPLIT_OUT/'split_boxes.csv',index=False)
full_mapping.to_csv(SPLIT_OUT/'all_images_split_mapping.csv',index=False)
split_summary.to_csv(SPLIT_OUT/'split_summary.csv')
class_split.to_csv(SPLIT_OUT/'class_distribution_by_split.csv')
for split in SPLIT_NAMES:
    (SPLIT_OUT/f'{split}.txt').write_text(''.join(p+'\n' for p in manifest.loc[manifest['split'].eq(split),'image']),encoding='utf-8')
# Fingerprint gồm danh sách ảnh, hash, nhóm, split và bbox; dùng so sánh các lần chạy.
payload={
    'images':manifest[['image','sha256','group_id','split']].sort_values('image').to_dict('records'),
    'boxes':manifest_boxes[['image','class_name','x1','y1','x2','y2']].sort_values(['image','class_name','x1','y1','x2','y2']).to_dict('records')
}
fingerprint=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
config={'seed':SPLIT_SEED,'target_ratios':dict(zip(SPLIT_NAMES,TARGET_RATIOS.tolist())),
        'board_regex':BOARD_REGEX,'label_atol_pixels':LABEL_ATOL,'input_images':len(images_df),
        'retained_images':len(manifest),
        'removed_extra_copies':int((~dedup_df.keep & ~dedup_df.quarantined).sum()),
        'conflict_policy':CONFLICT_POLICY,'quarantined_images':len(quarantine_df),
        'independent_groups':len(group_ids),'fingerprint':fingerprint,'split_score':split_score}
(SPLIT_OUT/'split_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
report_names=['dedup_decisions.csv','duplicate_label_conflicts.csv','quarantined_images.csv','quarantined_boxes.csv','all_image_groups.csv',
              'class_group_counts.csv','split_manifest.csv','split_boxes.csv','all_images_split_mapping.csv',
              'split_summary.csv','class_distribution_by_split.csv','split_distribution.png',
              'split_config.json','train.txt','val.txt','test.txt']
archive=SPLIT_OUT.parent/'pcb_prepare_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in report_names: z.write(SPLIT_OUT/name,arcname=name)
print('Fingerprint:',fingerprint)
print('Đã xuất:',archive,'— chỉ báo cáo và manifest, không chứa bộ ảnh.')
try:
    from IPython.display import FileLink
    display(FileLink(str(archive)))
except ImportError:
    pass


## 19. Xuất bộ dữ liệu baseline để bàn giao nhóm

Chạy sau mục 18 trong cùng kernel. Code sao chép ảnh được giữ, xuất nhãn YOLO,
thêm metadata, hướng dẫn và script kiểm tra; không thêm crop/augmentation, không train.
Giữ cách chia đã kiểm tra: 10.612 ảnh, 21.592 bbox; 56 ảnh xung đột được cách ly.
Test hiện chứa một board theo ID tên file; kết quả không đại diện mọi thiết kế PCB.

`EXPECTED_SPLIT_FINGERPRINT` khóa đúng manifest đã kiểm tra. Nếu không khớp, code
dừng để tránh xuất một cách chia khác mà không nhận biết. Giữ nguyên vị trí input
như lần chạy trước; fingerprint hiện có tính cả đường dẫn nguồn.


In [ ]:
import shutil, hashlib, json, csv, zipfile, importlib.util, sys
from pathlib import Path
from PIL import Image

EXPECTED_SPLIT_FINGERPRINT='b1aafdd0aa23fa30b74e77940c04046c47dad786d3b785a737c6547b988e2089'
PACKAGE_BASE=Path('/kaggle/working')
PACKAGE=PACKAGE_BASE/'PCB_Baseline_v1'
STAGING=PACKAGE_BASE/'PCB_Baseline_v1_building'
CLASS_ORDER=['missing_hole','mouse_bite','open_circuit','short','spur','spurious_copper']
assert set(manifest_boxes.class_name)==set(CLASS_ORDER)
export_payload={
    'images':manifest[['image','sha256','group_id','split']].sort_values('image').to_dict('records'),
    'boxes':manifest_boxes[['image','class_name','x1','y1','x2','y2']].sort_values(['image','class_name','x1','y1','x2','y2']).to_dict('records')
}
export_fingerprint=hashlib.sha256(json.dumps(export_payload,sort_keys=True).encode()).hexdigest()
assert export_fingerprint==EXPECTED_SPLIT_FINGERPRINT, 'Manifest khác bản đã kiểm tra. Không xuất; gửi split_config.json để đối chiếu.'
assert len(manifest)==10612 and len(manifest_boxes)==21592
assert not manifest.image.isin(quarantine_df.image).any()
missing_sources=[p for p in manifest.image if not Path(p).is_file()]
assert not missing_sources, f'Mất đường dẫn ảnh nguồn, ví dụ: {missing_sources[:3]}'
print('Đúng manifest đã khóa. Chuẩn bị xuất ảnh và nhãn YOLO.')


## 20. Sao chép ảnh và chuyển nhãn YOLO

Ảnh giữ nguyên byte. ID lớp 0–5 theo `CLASS_ORDER` ở trên; mỗi hàng nhãn là
`class cx cy width height` chuẩn hóa về [0,1]. Tọa độ pixel đầu vào đã được EDA
chuyển sang 0-based half-open, không trừ 1 thêm lần nữa. Kiểm tra ngược từ nhãn
YOLO về pixel cho từng bbox trước khi đóng gói.

Nếu thư mục đích đã có, code dừng để bảo vệ bản cũ. Dùng tên PACKAGE/STAGING khác
hoặc chỉ xóa thư mục output do chính notebook tạo nếu muốn xuất lại.


In [ ]:
assert not PACKAGE.exists(), f'{PACKAGE} đã tồn tại. Đổi PACKAGE và STAGING để xuất bản khác; không ghi đè.'
assert not STAGING.exists(), f'{STAGING} tồn tại từ lần chạy trước. Kiểm tra output trước khi xuất lại.'
STAGING.mkdir(parents=True)
for split in ['train','val','test']:
    (STAGING/'images'/split).mkdir(parents=True)
    (STAGING/'labels'/split).mkdir(parents=True)
for folder in ['metadata','reports','splits']:(STAGING/folder).mkdir()

def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

class_ids={name:i for i,name in enumerate(CLASS_ORDER)}
grouped_boxes={path:g for path,g in manifest_boxes.groupby('image',sort=False)}
export_rows=[]; export_box_rows=[]; used=set()
for r in tqdm(manifest.sort_values(['split','filename']).itertuples(),total=len(manifest),desc='Xuất dataset'):
    source=Path(r.image)
    # Prefix ID board và hash ngắn giúp tên xuất rõ nguồn và tránh va chạm.
    name=f'{r.board_id}_{r.sha256[:12]}_{source.name}'
    image_rel=Path('images')/r.split/name
    label_rel=Path('labels')/r.split/(Path(name).stem+'.txt')
    assert str(label_rel) not in used, 'Trùng tên nhãn xuất';used.add(str(label_rel))
    with Image.open(source) as im:width,height=im.size
    assert (width,height)==(int(r.width),int(r.height))
    shutil.copyfile(source,STAGING/image_rel)
    assert file_sha256(STAGING/image_rel)==r.sha256, f'Hash khác nguồn: {source}'
    subset=grouped_boxes.get(r.image,manifest_boxes.iloc[:0])
    lines=[]
    for b in subset.sort_values(['class_name','x1','y1','x2','y2']).itertuples():
        assert np.isfinite([b.x1,b.y1,b.x2,b.y2]).all()
        assert 0<=b.x1<b.x2<=width and 0<=b.y1<b.y2<=height
        xywh=np.array([(b.x1+b.x2)/(2*width),(b.y1+b.y2)/(2*height),
                       (b.x2-b.x1)/width,(b.y2-b.y1)/height])
        values=[float(f'{v:.10f}') for v in xywh]
        x,y,w,h=values
        recovered=np.array([(x-w/2)*width,(y-h/2)*height,(x+w/2)*width,(y+h/2)*height])
        assert np.allclose(recovered,[b.x1,b.y1,b.x2,b.y2],rtol=0,atol=1e-5)
        lines.append(str(class_ids[b.class_name])+' '+' '.join(f'{v:.10f}' for v in values))
        export_box_rows.append(dict(image=image_rel.as_posix(),class_id=class_ids[b.class_name],
                                   class_name=b.class_name,x1=b.x1,y1=b.y1,x2=b.x2,y2=b.y2,split=r.split))
    (STAGING/label_rel).write_text(''.join(line+'\n' for line in lines),encoding='utf-8')
    export_rows.append(dict(image=image_rel.as_posix(),label=label_rel.as_posix(),source_name=source.name,
                            split=r.split,board_id=r.board_id,group_id=r.group_id,sha256=r.sha256,
                            width=width,height=height,boxes=len(lines)))
package_manifest=pd.DataFrame(export_rows)
package_manifest.to_csv(STAGING/'metadata/manifest.csv',index=False)
pd.DataFrame(export_box_rows).to_csv(STAGING/'metadata/boxes_pixels.csv',index=False)
for split in ['train','val','test']:
    (STAGING/'splits'/f'{split}.txt').write_text(''.join('../'+p+'\n' for p in package_manifest.loc[package_manifest['split'].eq(split),'image']))
print('Đã sao chép',len(package_manifest),'ảnh và xuất',len(export_box_rows),'bbox.')


## 21. Thêm hướng dẫn bàn giao và kiểm tra độc lập


In [ ]:
VERIFY_SCRIPT = '"""Verify an exported PCB dataset; Python standard library plus Pillow."""\nfrom pathlib import Path\nimport csv, json, hashlib, math\nfrom collections import defaultdict, Counter\nfrom PIL import Image\n\ndef digest(path):\n    h=hashlib.sha256()\n    with path.open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\ndef verify(root):\n    root=Path(root).resolve()\n    with (root/\'metadata/manifest.csv\').open(encoding=\'utf-8\',newline=\'\') as f:\n        rows=list(csv.DictReader(f))\n    spec=json.loads((root/\'dataset_info.json\').read_text(encoding=\'utf-8\'))\n    checksum_lines=(root/\'checksums.sha256\').read_text().splitlines()\n    for line in checksum_lines:\n        expected,rel=line.split(\'  \',1)\n        assert digest(root/rel)==expected,f\'Checksum mismatch: {rel}\'\n    seen_paths=set(); hash_splits=defaultdict(set);board_splits=defaultdict(set)\n    groups=defaultdict(set); classes=Counter(); counts=Counter();total=0\n    for r in rows:\n        assert r[\'split\'] in (\'train\',\'val\',\'test\')\n        assert r[\'image\'] not in seen_paths;seen_paths.add(r[\'image\'])\n        image=root/r[\'image\'];label=root/r[\'label\']\n        with Image.open(image) as im:\n            im.load();w,h=im.size\n        assert (w,h)==(int(r[\'width\']),int(r[\'height\']))\n        assert digest(image)==r[\'sha256\']\n        hash_splits[r[\'sha256\']].add(r[\'split\'])\n        board_splits[r[\'board_id\']].add(r[\'split\']);groups[r[\'group_id\']].add(r[\'split\'])\n        lines=label.read_text().splitlines()\n        assert len(lines)==int(r[\'boxes\'])\n        for line in lines:\n            parts=line.split();assert len(parts)==5,f\'Invalid YOLO row: {label}\'\n            cid=int(parts[0]);x,y,bw,bh=map(float,parts[1:])\n            assert 0<=cid<len(spec[\'classes\'])\n            assert all(math.isfinite(v) for v in [x,y,bw,bh])\n            assert 0<bw<=1 and 0<bh<=1\n            assert x-bw/2>=-1e-8 and y-bh/2>=-1e-8\n            assert x+bw/2<=1+1e-8 and y+bh/2<=1+1e-8\n            classes[(r[\'split\'],cid)]+=1;total+=1\n        counts[r[\'split\']]+=1\n    assert all(len(s)==1 for m in (hash_splits,board_splits,groups) for s in m.values())\n    actual_images={p.relative_to(root).as_posix() for p in (root/\'images\').rglob(\'*\') if p.is_file()}\n    actual_labels={p.relative_to(root).as_posix() for p in (root/\'labels\').rglob(\'*.txt\')}\n    assert actual_images==set(r[\'image\'] for r in rows),\'Unexpected/missing image files\'\n    assert actual_labels==set(r[\'label\'] for r in rows),\'Unexpected/missing label files\'\n    assert len(rows)==spec[\'images\'] and total==spec[\'boxes\']\n    for split in (\'train\',\'val\',\'test\'):\n        assert counts[split]>0\n        assert all(classes[(split,i)]>0 for i in range(len(spec[\'classes\'])))\n        listed=(root/\'splits\'/f\'{split}.txt\').read_text().splitlines()\n        assert set(listed)=={\'../\'+r[\'image\'] for r in rows if r[\'split\']==split}\n        assert len(listed)==counts[split]\n    result={\'status\':\'passed\',\'images\':len(rows),\'boxes\':total,\'split_images\':dict(counts),\n            \'split_fingerprint\':spec[\'split_fingerprint\'],\'checksummed_files\':len(checksum_lines)}\n    print(json.dumps(result,ensure_ascii=False,indent=2))\n    return result\n\nif __name__==\'__main__\':verify(Path(__file__).resolve().parent)\n'
BIND_SCRIPT = '"""Run once after moving/unpacking the dataset to rewrite data.yaml."""\nfrom pathlib import Path\nimport json\n\ndef bind(root):\n    root=Path(root).resolve()\n    info=json.loads((root/\'dataset_info.json\').read_text(encoding=\'utf-8\'))\n    text=\'path: \'+json.dumps(root.as_posix(),ensure_ascii=False)+\'\\n\'\n    text+=\'train: images/train\\nval: images/val\\ntest: images/test\\nnames:\\n\'\n    text+=\'\'.join(f\'  {i}: {name}\\n\' for i,name in enumerate(info[\'classes\']))\n    path=root/\'data.yaml\';path.write_text(text,encoding=\'utf-8\');print(path)\n    return path\n\nif __name__==\'__main__\':bind(Path(__file__).resolve().parent)\n'


In [ ]:
(STAGING/'verify_dataset.py').write_text(VERIFY_SCRIPT,encoding='utf-8')
(STAGING/'bind_paths.py').write_text(BIND_SCRIPT,encoding='utf-8')
report_files=['split_config.json','split_summary.csv','class_distribution_by_split.csv',
              'split_distribution.png','duplicate_label_conflicts.csv','quarantined_images.csv',
              'quarantined_boxes.csv','dedup_decisions.csv','all_images_split_mapping.csv','class_group_counts.csv']
for name in report_files:
    src=SPLIT_OUT/name
    assert src.is_file(), f'Thiếu báo cáo: {src}'
    shutil.copyfile(src,STAGING/'reports'/name)
info={'name':'PCB_Baseline_v1','images':len(package_manifest),'boxes':len(export_box_rows),
      'classes':CLASS_ORDER,'class_ids':class_ids,'split_fingerprint':export_fingerprint,
      'source':'HRIPCB/PKU augmented VOC_PCB supplied by user',
      'input_images':len(images_df),'quarantined_images':len(quarantine_df),
      'removed_consistent_duplicates':int((~dedup_df.keep & ~dedup_df.quarantined).sum()),
      'grouping':'board ID parsed from filename plus exact-hash connections',
      'coordinate_convention':'YOLO normalized xywh; pixel metadata uses 0-based half-open',
      'new_augmentation_added':False,'trained_models_included':False,
      'split_images':package_manifest['split'].value_counts().to_dict(),
      'split_boards':{s:sorted(package_manifest.loc[package_manifest['split'].eq(s),'board_id'].unique()) for s in ['train','val','test']},
      'limitations':['Board IDs are inferred from filenames, not independently confirmed.',
                    'Test contains one board; variants/crops are correlated.',
                    'No guarantee against visually similar images with different hashes.',
                    'Review original dataset terms before sharing outside the project.']}
(STAGING/'dataset_info.json').write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
readme=f"""# PCB baseline dùng chung cho nhóm

Gói gồm {len(package_manifest)} ảnh, {len(export_box_rows)} bbox, 6 lớp.
Train/val/test cố định theo reports/split_summary.csv. Không tự chia lại.
Ảnh trùng có nhãn xung đột đã bị cách ly khỏi cả 3 tập; không có trong images/.
reports/ giữ danh sách và nhãn liên quan để đối chiếu; ảnh gốc không nằm trong gói cách ly.

## Sử dụng sau khi giải nén trên Kaggle hoặc máy khác
1. Chạy: python /duong/dan/PCB_Baseline_v1/verify_dataset.py
2. Chạy: python /duong/dan/PCB_Baseline_v1/bind_paths.py
   Script tạo lại data.yaml theo đường dẫn máy hiện tại. Chạy lại khi di chuyển gói.
   Nếu dataset được mount chỉ đọc trên Kaggle, dùng đoạn Python bên dưới.
3. Dùng đường dẫn tuyệt đối đến data.yaml khi huấn luyện YOLO.

Kaggle input chỉ đọc: tạo YAML làm việc ở /kaggle/working:
```python
from pathlib import Path
import json
root = Path('/kaggle/input/TEN_DATASET/PCB_Baseline_v1').resolve()
info = json.loads((root/'dataset_info.json').read_text())
lines = ['path: '+json.dumps(str(root)), 'train: images/train', 'val: images/val', 'test: images/test', 'names:']
lines += [f'  {{i}}: {{name}}' for i,name in enumerate(info['classes'])]
Path('/kaggle/working/pcb_data.yaml').write_text(chr(10).join(lines)+chr(10))
```

## Quy ước nhãn
Mỗi dòng: class_id cx cy width height, tọa độ chuẩn hóa [0,1], class ID bắt đầu từ 0.
Tên và thứ tự lớp nằm trong dataset_info.json, không đổi ID giữa các thành viên.
metadata/manifest.csv dùng đường dẫn tương đối; metadata/boxes_pixels.csv giúp đối chiếu pixel.
Không trừ 1 thêm khi dùng bbox trong metadata.

## Quy tắc thí nghiệm
Mọi mô hình dùng cùng cách chia. Chọn mô hình và tham số bằng validation.
Chỉ đánh giá test sau khi chốt cấu hình. Không tạo augmentation cho validation/test.
Gói này giữ các biến thể augmented có sẵn của nguồn; chưa thêm crop cải tiến.
Phiên bản crop ưu tiên lỗi nhỏ sau này chỉ thay phần train, giữ val/test cố định.
Test chỉ có một board, nên cần ghi giới hạn này khi báo cáo; không xem các crop là board độc lập.
ID board suy từ tên file, cần xác nhận với nguồn khi làm kết luận tổng quát hóa.
Nên thống nhất kích thước đầu vào, ngân sách huấn luyện và cách đo tốc độ giữa thành viên.

## Tính toàn vẹn
Fingerprint chia tập: {export_fingerprint}
checksums.sha256 chứa hash ảnh, nhãn và tài liệu cố định.
data.yaml không nằm trong checksums vì cần cập nhật đường dẫn sau khi di chuyển.
Gói không chứa model và chưa có kết quả huấn luyện.
Nguồn: bộ VOC_PCB augmented được cung cấp cho dự án HRIPCB/PKU PCB.
Tham khảo nguồn gốc: https://github.com/Ixiaohuihuihui/Tiny-Defect-Detection-for-PCB
Giữ trích dẫn nguồn và kiểm tra điều kiện sử dụng dữ liệu khi chia sẻ.
"""
(STAGING/'README.md').write_text(readme,encoding='utf-8')
# Mutable YAML will be bound only after directory is in its final location.
fixed_files=sorted(p for p in STAGING.rglob('*') if p.is_file())
(STAGING/'checksums.sha256').write_text(''.join(file_sha256(p)+'  '+p.relative_to(STAGING).as_posix()+'\n' for p in fixed_files))
namespace={'__name__':'pcb_package_verify'}
exec(compile(VERIFY_SCRIPT,'verify_dataset.py','exec'),namespace)
export_validation=namespace['verify'](STAGING)
# Rename only after full validation; output directory must still be absent.
assert not PACKAGE.exists()
STAGING.rename(PACKAGE)
bind_namespace={'__name__':'pcb_package_bind'}
exec(compile(BIND_SCRIPT,'bind_paths.py','exec'),bind_namespace)
bind_namespace['bind'](PACKAGE)
print('Dataset hoàn chỉnh:',PACKAGE)


## 22. Đóng ZIP bàn giao và báo cáo kiểm tra gọn


In [ ]:
DATASET_ZIP=PACKAGE_BASE/(PACKAGE.name+'.zip')
assert not DATASET_ZIP.exists(), f'{DATASET_ZIP} đã có. Đổi tên output để giữ bản cũ.'
with zipfile.ZipFile(DATASET_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in tqdm(sorted(PACKAGE.rglob('*')),desc='Đóng ZIP'):
        if p.is_file():z.write(p,arcname=PACKAGE.name+'/'+p.relative_to(PACKAGE).as_posix())
with zipfile.ZipFile(DATASET_ZIP) as z:
    bad=z.testzip()
    assert bad is None, f'ZIP lỗi CRC: {bad}'
archive_sha=file_sha256(DATASET_ZIP)
(PACKAGE_BASE/(PACKAGE.name+'.zip.sha256')).write_text(archive_sha+'  '+DATASET_ZIP.name+'\n')
report={'validation':export_validation,'archive':DATASET_ZIP.name,
        'archive_bytes':DATASET_ZIP.stat().st_size,'archive_sha256':archive_sha,
        'note':'Manifest checked; source board IDs inferred from filenames.'}
HANDOFF_REPORT=PACKAGE_BASE/'pcb_handoff_validation.json'
HANDOFF_REPORT.write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(f'Gói bàn giao: {DATASET_ZIP} ({DATASET_ZIP.stat().st_size/2**30:.2f} GiB)')
print('Gửi PCB_Baseline_v1.zip cho nhóm. Gửi pcb_handoff_validation.json cho mình kiểm tra kết quả xuất.')
from IPython.display import FileLink, display
display(FileLink(str(DATASET_ZIP)))
display(FileLink(str(HANDOFF_REPORT)))
